Install needed packages

In [6]:
!pip install gradio==4.31.0 json5==0.9.25 jsonschema==4.21.1 pydantic==2.7.1 python-dotenv==1.0.1 pygments==2.18.0 huggingface-hub==0.19.4

json_refiner_core Section

In [7]:
%%writefile json_refiner_core.py



import json
import re
from enum import Enum
from typing import Any, Dict, List, Optional, Union, Tuple
from datetime import datetime
import json5
from jsonschema import validate, ValidationError
from pydantic import BaseModel, Field
import traceback


class CaseStyle(Enum):
    """Enum for supported case conversion styles"""
    SNAKE_CASE = "snake_case"
    CAMEL_CASE = "camelCase"
    KEBAB_CASE = "kebab-case"
    PASCAL_CASE = "PascalCase"


class TransformationRecord(BaseModel):
    """Pydantic model for transformation history records"""
    timestamp: str
    operation: str
    input_preview: str
    output_preview: str
    status: str
    details: Optional[Dict[str, Any]] = None


class JSONRefinerCore:
    """
    Core class for JSON refinement operations
    """

    def __init__(self):
        self.conversion_log: List[Dict[str, Any]] = []
        self.validation_errors: List[Dict[str, Any]] = []
        self.transformation_history: List[TransformationRecord] = []

    def _add_to_history(self, operation: str, input_data: Any, output_data: Any,
                        status: str, details: Optional[Dict] = None):
        """Add transformation to history"""
        try:
            record = TransformationRecord(
                timestamp=datetime.now().isoformat(),
                operation=operation,
                input_preview=str(input_data)[:100] + "..." if len(str(input_data)) > 100 else str(input_data),
                output_preview=str(output_data)[:100] + "..." if output_data and len(str(output_data)) > 100 else str(output_data),
                status=status,
                details=details
            )
            self.transformation_history.append(record)
        except Exception as e:
            print(f"Error adding to history: {e}")

    # ==================== TYPE INFERENCE ENGINE ====================

    def infer_type(self, value: str) -> Dict[str, Any]:
        """
        Infer the type of a string value
        Returns a dictionary with type information and converted value
        """
        value = value.strip()

        # Check for null values
        null_values = {'null', 'none', 'nil', 'undefined', 'n/a', ''}
        if value.lower() in null_values:
            return {"type": "null", "value": None}

        # Check for boolean values
        true_values = {'true', 'yes', 'on', '1'}
        false_values = {'false', 'no', 'off', '0'}
        if value.lower() in true_values:
            return {"type": "boolean", "value": True}
        if value.lower() in false_values:
            return {"type": "boolean", "value": False}

        # Check for integers
        try:
            int_val = int(value)
            return {"type": "integer", "value": int_val}
        except ValueError:
            pass

        # Check for floats
        try:
            float_val = float(value)
            return {"type": "float", "value": float_val}
        except ValueError:
            pass

        # Check for JSON arrays
        if value.startswith('[') and value.endswith(']'):
            try:
                array_val = json5.loads(value)
                if isinstance(array_val, list):
                    return {"type": "array", "value": array_val}
            except:
                pass

        # Check for JSON objects
        if value.startswith('{') and value.endswith('}'):
            try:
                obj_val = json5.loads(value)
                if isinstance(obj_val, dict):
                    return {"type": "object", "value": obj_val}
            except:
                pass

        # Default to string
        return {"type": "string", "value": value}

    # ==================== CASE CONVERSION SYSTEM ====================

    def _split_words(self, key: str) -> List[str]:
        """Split a key into words based on common separators"""
        # Replace special characters with spaces
        key = re.sub(r'[^a-zA-Z0-9]', ' ', key)
        # Split on capital letters (for camelCase and PascalCase)
        key = re.sub(r'([a-z])([A-Z])', r'\1 \2', key)
        # Split and filter empty strings
        words = [w.lower() for w in key.split() if w]
        return words

    def to_snake_case(self, key: str) -> str:
        """Convert to snake_case"""
        words = self._split_words(key)
        return '_'.join(words)

    def to_camel_case(self, key: str) -> str:
        """Convert to camelCase"""
        words = self._split_words(key)
        if not words:
            return ''
        return words[0] + ''.join(w.capitalize() for w in words[1:])

    def to_kebab_case(self, key: str) -> str:
        """Convert to kebab-case"""
        words = self._split_words(key)
        return '-'.join(words)

    def to_pascal_case(self, key: str) -> str:
        """Convert to PascalCase"""
        words = self._split_words(key)
        return ''.join(w.capitalize() for w in words)

    def normalize_key(self, key: str, style: CaseStyle) -> str:
        """Normalize a key to the specified case style"""
        conversion_map = {
            CaseStyle.SNAKE_CASE: self.to_snake_case,
            CaseStyle.CAMEL_CASE: self.to_camel_case,
            CaseStyle.KEBAB_CASE: self.to_kebab_case,
            CaseStyle.PASCAL_CASE: self.to_pascal_case
        }

        converter = conversion_map.get(style, self.to_snake_case)
        return converter(key)

    def convert_json_keys(self, data: Any, style: Union[CaseStyle, str]) -> Any:
        """Recursively convert all keys in a JSON object"""
        # Handle string input
        if isinstance(data, str):
            try:
                data = json5.loads(data)
            except:
                return data

        # Handle CaseStyle enum or string
        if isinstance(style, str):
            try:
                style = CaseStyle(style)
            except ValueError:
                style = CaseStyle.SNAKE_CASE

        if isinstance(data, dict):
            new_dict = {}
            for key, value in data.items():
                new_key = self.normalize_key(key, style)
                new_dict[new_key] = self.convert_json_keys(value, style)
            return new_dict
        elif isinstance(data, list):
            return [self.convert_json_keys(item, style) for item in data]
        else:
            return data

    # ==================== VALIDATION SYSTEM ====================

    def validate_json_schema(self, json_data: str, schema: str, required_fields: str = "") -> Dict[str, Any]:
        """
        Validate JSON data against a JSON schema
        Returns a validation report
        """
        report = {
            "valid": True,
            "errors": [],
            "warnings": []
        }

        try:
            # Parse inputs
            if isinstance(json_data, str):
                json_data = json5.loads(json_data)
            if isinstance(schema, str):
                schema = json5.loads(schema)

            # Validate against schema
            validate(instance=json_data, schema=schema)

            # Check required fields if provided
            if required_fields:
                field_list = [f.strip() for f in required_fields.split(',') if f.strip()]
                if field_list:
                    required_check = self.check_required_fields(json_data, field_list)
                    if not required_check["valid"]:
                        report["valid"] = False
                        report["errors"].extend([
                            {
                                "path": field,
                                "message": "Required field missing",
                                "validator": "required"
                            } for field in required_check["missing_fields"]
                        ])

        except ValidationError as e:
            report["valid"] = False
            error_path = list(e.path) if e.path else ["root"]
            report["errors"].append({
                "path": " -> ".join(str(p) for p in error_path),
                "message": e.message,
                "validator": e.validator
            })
        except Exception as e:
            report["valid"] = False
            report["errors"].append({
                "path": "root",
                "message": str(e),
                "validator": "unknown"
            })

        return report

    def check_required_fields(self, json_data: Dict, required_fields: List[str]) -> Dict[str, Any]:
        """
        Check if all required fields exist in the JSON data
        Supports dot notation for nested fields
        """
        report = {
            "valid": True,
            "missing_fields": [],
            "present_fields": []
        }

        for field in required_fields:
            # Handle nested fields with dot notation
            parts = field.split('.')
            current = json_data
            found = True

            for part in parts:
                if isinstance(current, dict) and part in current:
                    current = current[part]
                else:
                    found = False
                    break

            if found:
                report["present_fields"].append(field)
            else:
                report["valid"] = False
                report["missing_fields"].append(field)

        return report

    # ==================== JSON TRANSFORMATIONS ====================

    def flatten_json(self, data: Any, parent_key: str = '', sep: str = '.') -> Dict:
        """
        Flatten a nested JSON object into a single-level dictionary
        """
        # Handle string input
        if isinstance(data, str):
            try:
                data = json5.loads(data)
            except:
                return {"error": "Invalid JSON string"}

        items = []

        if isinstance(data, dict):
            for key, value in data.items():
                new_key = f"{parent_key}{sep}{key}" if parent_key else key
                if isinstance(value, dict):
                    items.extend(self.flatten_json(value, new_key, sep=sep).items())
                elif isinstance(value, list):
                    # Handle lists by indexing
                    for i, item in enumerate(value):
                        list_key = f"{new_key}{sep}{i}"
                        if isinstance(item, (dict, list)):
                            items.extend(self.flatten_json(item, list_key, sep=sep).items())
                        else:
                            items.append((list_key, item))
                else:
                    items.append((new_key, value))
        elif isinstance(data, list):
            for i, item in enumerate(data):
                new_key = f"{parent_key}{sep}{i}" if parent_key else str(i)
                if isinstance(item, (dict, list)):
                    items.extend(self.flatten_json(item, new_key, sep=sep).items())
                else:
                    items.append((new_key, item))
        else:
            items.append((parent_key, data))

        return dict(items)

    def unflatten_json(self, flat_data: Any, sep: str = '.') -> Dict:
        """
        Unflatten a flattened JSON object back to nested structure
        """
        # Handle string input
        if isinstance(flat_data, str):
            try:
                flat_data = json5.loads(flat_data)
            except:
                return {"error": "Invalid JSON string"}

        # Handle non-dict input
        if not isinstance(flat_data, dict):
            return flat_data

        result = {}

        for key, value in flat_data.items():
            parts = key.split(sep)
            current = result

            # Navigate to the correct nesting level
            for i, part in enumerate(parts[:-1]):
                # Check if part is a number (array index)
                if part.isdigit():
                    part = int(part)
                    if not isinstance(current, list):
                        # Convert to list if needed
                        temp = current
                        current = []
                        if temp:
                            current.append(temp)
                    # Ensure list has enough elements
                    while len(current) <= part:
                        current.append({} if i < len(parts)-2 else None)
                    if not isinstance(current[part], (dict, list)):
                        current[part] = {}
                    current = current[part]
                else:
                    if part not in current:
                        # Check if next part is a number to decide structure
                        next_part = parts[i + 1] if i + 1 < len(parts) else None
                        if next_part and next_part.isdigit():
                            current[part] = []
                        else:
                            current[part] = {}
                    current = current[part]

            # Set the final value
            last_part = parts[-1]
            if last_part.isdigit():
                last_part = int(last_part)
                if not isinstance(current, list):
                    # Convert to list if needed
                    temp = current
                    current = []
                    if temp:
                        current.append(temp)
                # Ensure list has enough elements
                while len(current) <= last_part:
                    current.append(None)
                current[last_part] = value
            else:
                current[last_part] = value

        return result

    def deep_merge_json(self, *json_strings: str) -> Dict:
        """
        Deep merge multiple JSON objects
        Later dictionaries overwrite earlier ones for conflicts
        """
        result = {}

        for json_str in json_strings:
            if not json_str or not json_str.strip():
                continue

            try:
                # Parse JSON string
                if isinstance(json_str, str):
                    d = json5.loads(json_str)
                else:
                    d = json_str

                if not isinstance(d, dict):
                    continue

                # Merge dictionaries
                for key, value in d.items():
                    if key in result:
                        if isinstance(result[key], dict) and isinstance(value, dict):
                            # Recursively merge dictionaries
                            result[key] = self.deep_merge_json(
                                json.dumps(result[key]),
                                json.dumps(value)
                            )
                        elif isinstance(result[key], list) and isinstance(value, list):
                            # Merge lists by concatenation
                            result[key] = result[key] + value
                        else:
                            # Overwrite with new value
                            result[key] = value
                    else:
                        result[key] = value

            except Exception as e:
                # Skip invalid JSON
                print(f"Error parsing JSON: {e}")
                continue

        return result

    def remove_null_values(self, data: Any) -> Any:
        """
        Recursively remove all null values (None) from JSON data
        """
        if isinstance(data, dict):
            return {k: self.remove_null_values(v) for k, v in data.items()
                    if v is not None and self.remove_null_values(v) is not None}
        elif isinstance(data, list):
            cleaned = [self.remove_null_values(item) for item in data
                      if item is not None and self.remove_null_values(item) is not None]
            return cleaned
        else:
            return data

    def generate_statistics(self, data: Any) -> Dict[str, Any]:
        """
        Generate comprehensive statistics about the JSON data
        """
        stats = {
            "total_keys": 0,
            "depth": 0,
            "data_types": {},
            "null_values": 0,
            "arrays": 0,
            "objects": 0,
            "strings": 0,
            "numbers": 0,
            "booleans": 0,
            "sample_values": {}
        }

        def _traverse(obj, current_depth=1):
            stats["depth"] = max(stats["depth"], current_depth)

            if isinstance(obj, dict):
                stats["objects"] += 1
                for key, value in obj.items():
                    stats["total_keys"] += 1

                    # Track data types
                    value_type = type(value).__name__
                    stats["data_types"][value_type] = stats["data_types"].get(value_type, 0) + 1

                    # Track specific types
                    if value is None:
                        stats["null_values"] += 1
                    elif isinstance(value, str):
                        stats["strings"] += 1
                        if len(stats["sample_values"].get("strings", [])) < 3:
                            stats["sample_values"].setdefault("strings", []).append(value[:50])
                    elif isinstance(value, (int, float)):
                        stats["numbers"] += 1
                    elif isinstance(value, bool):
                        stats["booleans"] += 1
                    elif isinstance(value, list):
                        stats["arrays"] += 1

                    _traverse(value, current_depth + 1)

            elif isinstance(obj, list):
                stats["arrays"] += 1
                for item in obj:
                    _traverse(item, current_depth + 1)

        _traverse(data)
        return stats

    # ==================== HISTORY MANAGEMENT ====================

    def get_history(self) -> List[List[str]]:
        """Return the transformation history as a list of lists for Dataframe."""
        history_list = []
        for record in self.transformation_history:
            history_list.append([
                record.timestamp,
                record.operation,
                record.status,
                record.input_preview,
                record.output_preview
            ])
        return history_list

    def clear_history(self) -> List[List[str]]:
        """Clear the transformation history and return an empty list."""
        self.transformation_history.clear()
        return []

    def download_history(self) -> str:
        """Download the transformation history as a JSON file."""
        history_data = [record.model_dump() for record in self.transformation_history]
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"json_refiner_history_{timestamp}.json"
        with open(filename, "w") as f:
            json.dump(history_data, f, indent=2)
        return filename

    # ==================== MAIN REFINEMENT PIPELINE ====================

    def refine_json(self, input_data: str, case_style: str,
                   remove_nulls: bool = False, flatten: bool = False) -> Tuple[Dict, Dict, str]:
        """
        Main pipeline for JSON refinement
        Returns: (output_json, statistics, status_message)
        """
        try:
            # Parse input data
            if input_data.strip().startswith('{'):
                data = json5.loads(input_data)
            else:
                # Handle key-value pairs
                data = {}
                for line in input_data.strip().split('\n'):
                    if ':' in line:
                        key, value = line.split(':', 1)
                        inferred = self.infer_type(value.strip())
                        data[key.strip()] = inferred["value"]

            # Apply transformations
            transformed = data.copy()

            # Convert case style
            if case_style != CaseStyle.SNAKE_CASE.value:
                transformed = self.convert_json_keys(transformed, case_style)

            # Remove nulls if requested
            if remove_nulls:
                transformed = self.remove_null_values(transformed)

            # Flatten if requested
            if flatten:
                transformed = self.flatten_json(transformed)

            # Generate statistics
            stats = self.generate_statistics(transformed)

            # Add to history
            self._add_to_history(
                operation="refine",
                input_data=data,
                output_data=transformed,
                status="success",
                details={
                    "case_style": case_style,
                    "remove_nulls": remove_nulls,
                    "flatten": flatten
                }
            )

            return transformed, stats, "✅ Success"

        except Exception as e:
            error_msg = f"❌ Error: {str(e)}"
            self._add_to_history(
                operation="refine",
                input_data=input_data,
                output_data=None,
                status="error",
                details={"error": str(e)}
            )
            return {"error": str(e)}, {}, error_msg

Overwriting json_refiner_core.py


gradio UI designing

In [19]:
%%writefile gradio_ui.py

import gradio as gr
import json
from typing import List, Dict, Any
from datetime import datetime
from json_refiner_core import JSONRefinerCore, CaseStyle
import traceback


class JSONRefinerUI:
    """
    Gradio UI for JSON Refiner application
    """

    def __init__(self):
        self.core = JSONRefinerCore()

    def create_app(self):
        """Create the Gradio interface with 7 tabs"""

        with gr.Blocks(theme=gr.themes.Soft(), title="JSON Refiner - Advanced Edition") as app:

            gr.Markdown("""
            # 🔧 JSON Refiner – Advanced Edition
            ### Professional JSON Processing Platform
            """)

            with gr.Tabs():

                # ========== TAB 1: Core Refining ==========
with gr.TabItem("🔄 Core Refining", id=1):
    with gr.Row():

        with gr.Column(scale=1):
            gr.Markdown("### Input")

            input_text = gr.Textbox(
                label="Key-Value Pairs or JSON",
                placeholder="Enter key: value pairs (one per line) or paste JSON",
                lines=10,
                value="name: John Doe\nage: 30\nemail: john@example.com\nactive: true"
            )

            case_style = gr.Dropdown(
                label="Case Style",
                choices=[style.value for style in CaseStyle],
                value=CaseStyle.SNAKE_CASE.value
            )

            with gr.Row():
                remove_nulls = gr.Checkbox(label="Remove Null Values", value=False)
                flatten = gr.Checkbox(label="Flatten JSON", value=False)

            refine_btn = gr.Button("✨ Refine JSON", variant="primary")

        with gr.Column(scale=1):
            gr.Markdown("### Output")

            output_json = gr.JSON(label="Refined JSON")
            status_text = gr.Markdown("✅ Ready")
            statistics = gr.JSON(label="Statistics")

    refine_btn.click(
        fn=self.core.refine_json,
        inputs=[input_text, case_style, remove_nulls, flatten],
        outputs=[output_json, statistics, status_text]
    )



                # ========== TAB 2: Validation ==========
                with gr.TabItem("✅ Validation", id=2):
                    with gr.Row():

                        with gr.Column():
                            validate_input = gr.Textbox(
                                label="JSON to Validate",
                                lines=8,
                                value='{"name": "John", "age": 30, "email": "john@example.com"}'
                            )

                            schema_input = gr.Textbox(
                                label="JSON Schema (Draft 7)",
                                lines=8,
                                value='{"type": "object", "properties": {"name": {"type": "string"}, "age": {"type": "integer"}, "email": {"type": "string"}}, "required": ["name", "email"]}'
                            )

                            required_input = gr.Textbox(
                                label="Required Fields (comma-separated)",
                                value="name, email",
                                lines=2
                            )

                            validate_btn = gr.Button("🔍 Validate", variant="primary")

                        with gr.Column():
                            validation_result = gr.JSON(label="Validation Report")

                    validate_btn.click(
                        fn=self.core.validate_json_schema,
                        inputs=[validate_input, schema_input, required_input],
                        outputs=[validation_result]
                    )

                # ========== TAB 3: Case Conversion ==========
                with gr.TabItem("🔄 Case Conversion", id=3):
                    with gr.Row():

                        with gr.Column():
                            case_input = gr.Textbox(
                                label="JSON Input",
                                lines=10,
                                value='{"user_name": "John Doe", "user_age": 30, "user_email": "john@example.com"}'
                            )

                            target_case = gr.Dropdown(
                                label="Target Case",
                                choices=[style.value for style in CaseStyle],
                                value=CaseStyle.CAMEL_CASE.value
                            )

                            convert_btn = gr.Button("🔄 Convert", variant="primary")

                        with gr.Column():
                            case_output = gr.JSON(label="Converted JSON")

                    convert_btn.click(
                        fn=self.core.convert_json_keys,
                        inputs=[case_input, target_case],
                        outputs=[case_output]
                    )

                # ========== TAB 4: Merge JSON ==========
                with gr.TabItem("🔄 Merge JSON", id=4):
                    with gr.Row():

                        with gr.Column():
                            json1 = gr.Textbox(label="JSON 1", value='{"a": 1, "b": {"c": 2}}', lines=5)
                            json2 = gr.Textbox(label="JSON 2", value='{"b": {"d": 3}, "e": 4}', lines=5)
                            json3 = gr.Textbox(label="JSON 3 (Optional)", value='{"f": 5, "a": 10}', lines=5)

                            merge_btn = gr.Button("🔗 Merge", variant="primary")

                        with gr.Column():
                            merged_output = gr.JSON(label="Merged JSON")

                    merge_btn.click(
                        fn=self.core.deep_merge_json,
                        inputs=[json1, json2, json3],
                        outputs=[merged_output]
                    )

                # ========== TAB 5: Flatten / Unflatten ==========
                with gr.TabItem("📊 Flatten / Unflatten", id=5):
                    with gr.Row():

                        with gr.Column():
                            flat_input = gr.Textbox(
                                label="JSON Input",
                                lines=10,
                                value='{"user": {"name": "John", "address": {"city": "NYC", "zip": 10001}, "hobbies": ["reading", "gaming"]}}'
                            )

                            with gr.Row():
                                flatten_btn = gr.Button("📉 Flatten", variant="primary")
                                unflatten_btn = gr.Button("📈 Unflatten", variant="secondary")

                            flat_output = gr.JSON(label="Result")

                    flatten_btn.click(
                        fn=self.core.flatten_json,
                        inputs=[flat_input],
                        outputs=[flat_output]
                    )

                    unflatten_btn.click(
                        fn=self.core.unflatten_json,
                        inputs=[flat_output],
                        outputs=[flat_output]
                    )

                # ========== TAB 6: History ==========
                with gr.TabItem("📜 History", id=6):
                    with gr.Row():

                        with gr.Column():
                            refresh_history_btn = gr.Button("🔄 Refresh History")
                            clear_history_btn = gr.Button("🗑️ Clear History")
                            download_history_btn = gr.Button("📥 Download Report", variant="primary")

                        with gr.Column():
                            history_display = gr.Dataframe(
                                label="Transformation History",
                                headers=["Timestamp", "Operation", "Status", "Input Preview", "Output Preview"],
                                interactive=False,
                                value=[]
                            )

                            history_output = gr.File(label="Download")

                    refresh_history_btn.click(
                        fn=self.core.get_history,
                        inputs=[],
                        outputs=[history_display]
                    )

                    clear_history_btn.click(
                        fn=self.core.clear_history,
                        inputs=[],
                        outputs=[history_display]
                    )

                    download_history_btn.click(
                        fn=self.core.download_history,
                        inputs=[],
                        outputs=[history_output]
                    )

                # ========== TAB 7: Guide ==========
                with gr.TabItem("📖 Guide", id=7):
                    gr.Markdown("""
                    # 📚 JSON Refiner - User Guide

                    ## 🎯 Overview
                    JSON Refiner is a professional JSON processing platform for transforming, validating, and analyzing JSON data.

                    ## 🔧 Features
                    - Automatic Type Inference
                    - Case Style Conversion
                    - Schema Validation (Draft 7)
                    - Required Field Checking
                    - Deep Merge
                    - Flatten / Unflatten
                    - History Tracking

                    ## 💡 Example Input
                    name: John Doe
                    age: 30
                    is_active: true
                    address: {"city": "New York", "zip": "10001"}

                    ## ⚠️ Troubleshooting
                    - Ensure JSON is properly formatted
                    - Schema must follow Draft 7
                    - Large files may take a few seconds

                    ## 🚀 Tips
                    - Use Core Refining for quick transformations
                    - Check statistics for structure insight
                    - Download reports for documentation
                    """)

            gr.Markdown("🛠️ JSON Refiner - Advanced Edition | Built with Gradio 4.0+")

        return app

Overwriting gradio_ui.py


deploying UI to gradio

In [20]:
import gradio as gr
from gradio_ui import JSONRefinerUI
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

def main():
    """Main entry point for the application"""

    print("="*60)
    print("JSON REFINER - Advanced Edition")
    print("="*60)
    print("\nStarting application...")
    print("✓ Core module loaded")
    print("✓ UI module loaded")
    print("✓ Dependencies verified")

    # Create UI instance
    ui = JSONRefinerUI()

    # Create the Gradio app
    app = ui.create_app()

    # Configure launch settings
    launch_kwargs = {
        "server_name": "0.0.0.0",  # Allow external connections
        "server_port": 7320,        # Default Gradio port
        "share": True,               # Create public link
        "debug": False,              # Disable debug in production
        "show_error": True,          # Show errors in UI
        "quiet": False               # Show launch messages
    }

    # Override with environment variables if set
    if os.getenv("GRADIO_SERVER_NAME"):
        launch_kwargs["server_name"] = os.getenv("GRADIO_SERVER_NAME")
    if os.getenv("GRADIO_SERVER_PORT"):
        launch_kwargs["server_port"] = int(os.getenv("GRADIO_SERVER_PORT"))
    if os.getenv("GRADIO_SHARE", "").lower() == "false":
        launch_kwargs["share"] = False

    print(f"\n✓ Launch configuration ready")
    print(f"  Server: {launch_kwargs['server_name']}:{launch_kwargs['server_port']}")
    print(f"  Public link: {'Yes' if launch_kwargs['share'] else 'No'}")
    print("\n" + "="*60)
    print("Launching application...")
    print("="*60 + "\n")

    # Launch the app
    app.launch(**launch_kwargs)

if __name__ == "__main__":
    main()

JSON REFINER - Advanced Edition

Starting application...
✓ Core module loaded
✓ UI module loaded
✓ Dependencies verified

✓ Launch configuration ready
  Server: 0.0.0.0:7320
  Public link: Yes

Launching application...

IMPORTANT: You are using gradio version 4.31.0, however version 4.44.1 is available, please upgrade.
--------
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://b95cab9034b813b39c.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
